# Notebook 02 — Data Merging & Quality Gates

**Purpose:**  
Concatenate the four per-portal datasets into two unified tables. Apply cross-portal
quality rules and backfill the `ministry` column.

**Inputs** (`data/processed/`): 8 per-portal files from Notebook 01.

**Outputs** (`data/processed/`):
- `merged_bid_data.xlsx`
- `merged_financial_evaluation.xlsx`

**Quality rules applied:**
1. Drop rows where exactly one of `winner` / `winning_price` is null.
2. Drop rows where `bid_start_date` is missing.
3. Backfill `ministry` from `MINISTRY_MAP` via `department_org`.
4. Drop `created_at` and `Unnamed: 0` artefacts.

---

## 1. Imports & Setup

In [1]:
import sys
import pandas as pd
import numpy as np
from pathlib import Path

sys.path.insert(0, str(Path('..').resolve()))

from config import PROCESSED_FILES
from src.cleaning import clean_company_name
from src.schema import MINISTRY_MAP

print('Setup complete.')

Setup complete.


## 2. Load Per-Portal Files

In [2]:
portal_bid_files = {
    'GEM'       : PROCESSED_FILES['gem_bids'],
    'E Procure' : PROCESSED_FILES['eprocure_bids'],
    'E Tender'  : PROCESSED_FILES['etender_bids'],
    'Coal India': PROCESSED_FILES['coal_india_bids'],
}

portal_fe_files = {
    'GEM'       : PROCESSED_FILES['gem_fe'],
    'E Procure' : PROCESSED_FILES['eprocure_fe'],
    'E Tender'  : PROCESSED_FILES['etender_fe'],
    'Coal India': PROCESSED_FILES['coal_india_fe'],
}

bid_frames, fe_frames = [], []

for portal, path in portal_bid_files.items():
    df = pd.read_excel(path)
    df['website_name'] = portal
    bid_frames.append(df)
    print(f"  Loaded {len(df):>5} bid rows  from {portal}")

for portal, path in portal_fe_files.items():
    df = pd.read_excel(path)
    fe_frames.append(df)
    print(f"  Loaded {len(df):>5} FE rows   from {portal}")

  Loaded  3763 bid rows  from GEM
  Loaded   415 bid rows  from E Procure
  Loaded   292 bid rows  from E Tender
  Loaded  1220 bid rows  from Coal India
  Loaded 16199 FE rows   from GEM
  Loaded  1998 FE rows   from E Procure
  Loaded  1049 FE rows   from E Tender
  Loaded  3281 FE rows   from Coal India


## 3. Concatenate & Basic Cleaning

In [3]:
bid_data = pd.concat(bid_frames, ignore_index=True).drop_duplicates()
fe_data  = pd.concat(fe_frames,  ignore_index=True).drop_duplicates()

# Parse date columns after Excel round-trip
bid_data['bid_start_date'] = pd.to_datetime(bid_data['bid_start_date'], errors='coerce')
bid_data['bid_end_date']   = pd.to_datetime(bid_data['bid_end_date'],   errors='coerce')

# Clean winner / seller names here, once, immediately after merging all portals.
# clean_company_name is idempotent, so re-applying it to already-cleaned GEM/
# eProcure/eTender/Coal India winners (cleaned individually in Notebook 01) is
# safe and has no effect on them -- this pass exists specifically to normalise
# fe_data['seller'], which is NOT cleaned anywhere in Notebook 01.
# Downstream notebooks (03, 04) must not assume this column needs cleaning
# again -- if they do, check here first rather than re-cleaning blindly.
bid_data['winner'] = bid_data['winner'].apply(clean_company_name)
fe_data['seller']  = fe_data['seller'].astype(str).str.strip().apply(clean_company_name)

# Drop auto-index and timestamp artefacts from previous exports
bid_data.drop(columns=['Unnamed: 0', 'created_at'], inplace=True, errors='ignore')
fe_data.drop( columns=['Unnamed: 0'],               inplace=True, errors='ignore')

print(f"Merged bid_data : {bid_data.shape}")
print(f"Merged fe_data  : {fe_data.shape}")

Merged bid_data : (5690, 13)
Merged fe_data  : (22525, 5)


## 4. Duplicate bid_number Detection — CRITICAL

A `bid_number` must be unique across the merged dataset. If the same tender appears
more than once (re-scraped, re-published, or a stale snapshot mixed with a fresh one),
downstream steps that key on `bid_number` — `.map()`, `.set_index()`, Excel round-trips —
can silently misalign `winner` against the wrong tender. This has caused real data
corruption in this pipeline before, so it is checked explicitly here rather than assumed away.

**Rule:** if duplicate `bid_number`s disagree on `winner`, we cannot silently pick one —
that is exactly the kind of silent corruption this check exists to prevent. Conflicting
duplicates are logged for manual review and the most recently published row
(`bid_start_date` max) is kept as the best-guess resolution.

In [4]:
dupe_mask = bid_data.duplicated(subset='bid_number', keep=False)
dupes     = bid_data[dupe_mask].sort_values('bid_number')

print(f"Rows with a duplicated bid_number : {len(dupes)}")
print(f"Distinct bid_numbers affected      : {dupes['bid_number'].nunique()}")

if len(dupes) > 0:
    # Flag duplicates where winner disagrees across rows -- these are the dangerous ones
    conflicting = (
        dupes.groupby('bid_number')['winner']
        .apply(lambda s: s.nunique(dropna=True) > 1)
    )
    conflicting_ids = conflicting[conflicting].index.tolist()

    print(f"bid_numbers with CONFLICTING winner values : {len(conflicting_ids)}")
    if conflicting_ids:
        print("\nExample conflicts (first 10):")
        print(
            dupes[dupes['bid_number'].isin(conflicting_ids[:10])]
            [['bid_number', 'winner', 'winning_price', 'website_name', 'bid_start_date']]
            .to_string(index=False)
        )

    # Resolution: keep the most recently published row per bid_number.
    # This is a deliberate, auditable choice -- not a silent pandas default.
    bid_data = (
        bid_data
        .sort_values('bid_start_date', ascending=False)
        .drop_duplicates(subset='bid_number', keep='first')
        .reset_index(drop=True)
    )

    print(f"\nRows after deduplication on bid_number : {len(bid_data)}")
else:
    print("No duplicate bid_numbers found -- bid_number is a valid unique key.")

# Hard assertion -- nothing downstream may proceed on a non-unique key
assert bid_data['bid_number'].is_unique, "bid_number is still not unique after deduplication."
print("\nValidated: bid_number is now a unique key.")

Rows with a duplicated bid_number : 0
Distinct bid_numbers affected      : 0
No duplicate bid_numbers found -- bid_number is a valid unique key.

Validated: bid_number is now a unique key.


## 5. Data Quality Gates

Removes structurally invalid rows that would corrupt downstream analysis.

In [5]:
n_start = len(bid_data)

# Rule 1: winner / winning_price must both be present or both absent
inconsistent_mask = bid_data['winner'].isna() != bid_data['winning_price'].isna()
n_inconsistent    = inconsistent_mask.sum()
bid_data          = bid_data[~inconsistent_mask].copy()
print(f"Dropped {n_inconsistent:>4} rows — inconsistent winner / winning_price nullity")

# Rule 2: bid_start_date must be present
n_no_date = bid_data['bid_start_date'].isna().sum()
bid_data  = bid_data.dropna(subset=['bid_start_date']).copy()
print(f"Dropped {n_no_date:>4} rows — missing bid_start_date")

print(f"\nRows before quality gates : {n_start}")
print(f"Rows after  quality gates : {len(bid_data)}")

Dropped    0 rows — inconsistent winner / winning_price nullity
Dropped    1 rows — missing bid_start_date

Rows before quality gates : 5690
Rows after  quality gates : 5689


## 6. Ministry Backfill

eProcure and eTender do not include a ministry column in their exports.
`MINISTRY_MAP` in `src/schema.py` maps known department names to their parent ministries.

In [6]:
n_before = bid_data['ministry'].isna().sum()

bid_data['ministry'] = bid_data['ministry'].fillna(
    bid_data['department_org'].map(MINISTRY_MAP)
)

n_after = bid_data['ministry'].isna().sum()
print(f"Ministry null before backfill : {n_before}")
print(f"Ministry null after  backfill : {n_after}")
print(f"Ministries resolved           : {n_before - n_after}")

Ministry null before backfill : 706
Ministry null after  backfill : 0
Ministries resolved           : 706


## 7. Records Per Portal — Sanity Check

In [7]:
portal_summary = (
    bid_data.groupby('website_name')
    .agg(
        total_bids   =('bid_number', 'count'),
        awarded      =('winner', lambda x: x.notna().sum()),
        with_ministry=('ministry', lambda x: x.notna().sum()),
    )
    .assign(award_rate_pct=lambda d: (d['awarded'] / d['total_bids'] * 100).round(1))
)
print(portal_summary.to_string())

              total_bids  awarded  with_ministry  award_rate_pct
website_name                                                    
Coal India          1220     1192           1220            97.7
E Procure            414      264            414            63.8
E Tender             292       51            292            17.5
GEM                 3763     3713           3763            98.7


## 7b. Financial Evaluation Integrity Check

`financial_eval` legitimately has multiple rows per `bid_number` -- one per bidder.
What must NOT happen is the same `(bid_number, seller)` pair appearing twice,
which would double-count a bidder and could feed bad data into the seller master
and competitor analysis in later notebooks.

In [8]:
fe_dupe_mask = fe_data.duplicated(subset=['bid_number', 'seller'], keep=False)
fe_dupes     = fe_data[fe_dupe_mask]

print(f"Duplicate (bid_number, seller) pairs in financial_eval: {len(fe_dupes)}")

if len(fe_dupes) > 0:
    print("\nExample duplicate bidder entries (first 10 rows):")
    print(fe_dupes.sort_values(['bid_number', 'seller']).head(10).to_string(index=False))
    fe_data = fe_data.drop_duplicates(subset=['bid_number', 'seller'], keep='first').reset_index(drop=True)
    print(f"\nRows after deduplication: {len(fe_data)}")
else:
    print("No duplicate bidder entries found.")

Duplicate (bid_number, seller) pairs in financial_eval: 12

Example duplicate bidder entries (first 10 rows):
        bid_number rank_position                                          seller        price        status
  2020_HCL_40638_1            L1                               Varun Enterprises     85880.40           NaN
  2020_HCL_40638_1            L3                               Varun Enterprises     95261.40           NaN
 2022_ECL_238481_1           NaN                            Mukherjee Enterprise         1.00           NaN
 2022_ECL_238481_1           NaN                            Mukherjee Enterprise         2.00           NaN
GEM/2024/B/5002484            L1                                   Rajeev Ranjan   4435539.83 Not Evaluated
GEM/2024/B/5002484            L1                                   Rajeev Ranjan   4435539.83     Qualified
GEM/2024/B/5138422  DISQUALIFIED Prithvi Realcon & Transportation Privatelimited 242836294.08           NaN
GEM/2024/B/5138422        

## 8. Export

In [9]:
bid_data.to_excel(PROCESSED_FILES['merged_bids'], index=False)
fe_data.to_excel( PROCESSED_FILES['merged_fe'],   index=False)

print(f"Exported merged_bid_data           →  {PROCESSED_FILES['merged_bids'].name}  ({len(bid_data)} rows)")
print(f"Exported merged_financial_evaluation →  {PROCESSED_FILES['merged_fe'].name}  ({len(fe_data)} rows)")
print("\nNotebook 02 complete. Proceed to 03_feature_engineering.ipynb.")

Exported merged_bid_data           →  merged_bid_data.xlsx  (5689 rows)
Exported merged_financial_evaluation →  merged_financial_evaluation.xlsx  (22519 rows)

Notebook 02 complete. Proceed to 03_feature_engineering.ipynb.
